In [3]:
#Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [5]:
#LOAD DATA

In [6]:
df = pd.read_csv("../datas/btc_4h_full_features.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["year"] = df["timestamp"].dt.year

df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

,timestamp,open,high,low,close,volume,previous_close,tr1,tr2,tr3,true_range,atr_14,future_close,future_return,target,log_return,year
0,2017-08-19 08:00:00,4033.47,4082.25,3850.00,3957.60,77.256042,4033.47,232.25,48.78,183.47,232.25,161.687857,4106.53,0.037631,1,-0.018989,2017
1,2017-08-19 12:00:00,3945.12,4100.00,3928.89,3972.05,47.515578,3957.60,171.11,142.40,28.71,171.11,167.576429,4125.00,0.038507,1,0.003645,2017
2,2017-08-19 16:00:00,4036.19,4103.92,3953.40,4076.12,25.925056,3972.05,150.52,131.87,18.65,150.52,167.465714,4164.60,0.021707,0,0.025863,2017
3,2017-08-19 20:00:00,4088.71,4149.99,4051.47,4139.98,9.350560,4076.12,98.52,73.87,24.65,98.52,163.647857,4086.29,-0.012969,0,0.015545,2017
4,2017-08-20 00:00:00,4120.98,4139.98,4044.00,4094.62,6.611283,4139.98,95.98,0.00,95.98,95.98,159.496429,4084.76,-0.002408,0,-0.011017,2017


In [7]:
#DEFINE TAIL TARGET (5% WORST RETURNS)

In [8]:
tail_threshold = df["future_return"].quantile(0.05)

df["tail_event"] = (df["future_return"] <= tail_threshold).astype(int)

print("Tail threshold:", tail_threshold)
print("Baseline tail frequency:", df["tail_event"].mean())

Tail threshold: -0.05372132098633962
Baseline tail frequency: 0.05000803126840499


In [9]:
#DEFINE TAIL TARGET (5% WORST RETURNS)

In [10]:
df["tail_score_raw"] = np.nan

years = sorted(df["year"].unique())

for test_year in years[4:]:
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = df[df["year"].isin(train_years)]
    test_data = df[df["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[["atr_14"]]
    y_train = train_data["tail_event"]
    
    X_test = test_data[["atr_14"]]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    df.loc[df["year"] == test_year, "tail_score_raw"] = model.predict_proba(X_test_scaled)[:,1]

df[["timestamp", "tail_score_raw"]].tail()

,timestamp,tail_score_raw
18672,2026-02-28 00:00:00,0.016433
18673,2026-02-28 04:00:00,0.016732
18674,2026-02-28 08:00:00,0.016704
18675,2026-02-28 12:00:00,0.017078
18676,2026-02-28 16:00:00,0.017909


In [11]:
#SMOOTHING (EWMA)

In [12]:
df["tail_score_smooth"] = df["tail_score_raw"].ewm(span=20).mean()

df[["timestamp", "tail_score_smooth"]].tail()

,timestamp,tail_score_smooth
18672,2026-02-28 00:00:00,0.018049
18673,2026-02-28 04:00:00,0.017923
18674,2026-02-28 08:00:00,0.017807
18675,2026-02-28 12:00:00,0.017738
18676,2026-02-28 16:00:00,0.017754


In [13]:
#TAIL RISK INDEX (ROLLING PERCENTILE 365 PERIODS)

In [14]:
df["tail_index"] = (
    df["tail_score_smooth"]
    .rolling(365)
    .apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])
) * 100

df[["timestamp", "tail_index"]].tail()

,timestamp,tail_index
18672,2026-02-28 00:00:00,65.479452
18673,2026-02-28 04:00:00,64.657534
18674,2026-02-28 08:00:00,64.109589
18675,2026-02-28 12:00:00,63.835616
18676,2026-02-28 16:00:00,64.383562


In [15]:
#RISK LEVEL CLASSIFICATION

In [16]:
def risk_level(x):
    if pd.isna(x):
        return None
    elif x > 95:
        return "Extreme"
    elif x > 85:
        return "High"
    elif x > 70:
        return "Elevated"
    else:
        return "Normal"

df["risk_level"] = df["tail_index"].apply(risk_level)

df["risk_level"].value_counts()

risk_level
Normal      7946
Elevated    1338
High        1043
Extreme      618
Name: count, dtype: int64

In [17]:
#VALIDATION – EXTREME VS BASELINE

In [18]:
extreme = df[df["risk_level"] == "Extreme"]

print("Extreme count:", len(extreme))
print("Tail freq in Extreme:", extreme["tail_event"].mean())
print("Baseline tail freq:", df["tail_event"].mean())

Extreme count: 618
Tail freq in Extreme: 0.07605177993527508
Baseline tail freq: 0.05000803126840499


In [19]:
#EXPOSURE SCALING STRATEGY

In [20]:
df["position_size"] = 1.0

df.loc[df["risk_level"] == "Elevated", "position_size"] = 0.9
df.loc[df["risk_level"] == "High", "position_size"] = 0.7
df.loc[df["risk_level"] == "Extreme", "position_size"] = 0.5

df["strategy_return"] = df["position_size"] * df["log_return"]

df["equity_strategy"] = np.exp(df["strategy_return"].cumsum())
df["equity_buyhold"] = np.exp(df["log_return"].cumsum())

df[["equity_strategy", "equity_buyhold"]].tail()

,equity_strategy,equity_buyhold
18672,12.542964,16.312857
18673,12.167153,15.824092
18674,12.170622,15.828604
18675,12.360676,16.075781
18676,12.650592,16.452833


In [21]:
#MAX DRAWDOWN COMPARISON

In [22]:
def max_drawdown(series):
    cummax = series.cummax()
    drawdown = (series - cummax) / cummax
    return drawdown.min()

print("Buy & Hold MDD:", max_drawdown(df["equity_buyhold"]))
print("Strategy MDD:", max_drawdown(df["equity_strategy"]))

Buy & Hold MDD: -0.8390598442375499
Strategy MDD: -0.8390598442375499


In [23]:
#Calcolo Ema 200

In [24]:
df["ema_200"] = df["close"].ewm(span=200).mean()

In [25]:
#Calcolo slope ema 200

In [26]:
df["ema_200_slope"] = df["ema_200"].diff()

In [27]:
#Calcolo regime

In [28]:
def regime_state(row):
    if row["close"] > row["ema_200"] and row["ema_200_slope"] > 0:
        return "Bull"
    elif row["close"] < row["ema_200"] and row["ema_200_slope"] < 0:
        return "Bear"
    else:
        return "Neutral"

df["regime_state"] = df.apply(regime_state, axis=1)

df["regime_state"].value_counts()

regime_state
Bull       9762
Bear       8914
Neutral       1
Name: count, dtype: int64

In [29]:
#Regime exposure rule

In [30]:
df["regime_exposure"] = 1.0

df.loc[df["regime_state"] == "Neutral", "regime_exposure"] = 0.6
df.loc[df["regime_state"] == "Bear", "regime_exposure"] = 0.3

In [31]:
#Combinazione con tail layer

In [32]:
df["final_exposure"] = df["regime_exposure"]

df.loc[df["risk_level"] == "Elevated", "final_exposure"] *= 0.9
df.loc[df["risk_level"] == "High", "final_exposure"] *= 0.7
df.loc[df["risk_level"] == "Extreme", "final_exposure"] *= 0.5

In [33]:
# Shift exposure di 1 barra (decisione oggi → applicata alla barra dopo)
df["final_exposure_shifted"] = df["final_exposure"].shift(1)

# Ritorno strategia corretto
df["strategy_return_v2"] = df["final_exposure_shifted"] * df["log_return"]

# Equity curve corretta
df["equity_strategy_v2"] = np.exp(df["strategy_return_v2"].cumsum())

In [34]:
print("Buy & Hold MDD:", max_drawdown(df["equity_buyhold"]))
print("Strategy v2 MDD:", max_drawdown(df["equity_strategy_v2"]))

Buy & Hold MDD: -0.8390598442375499
Strategy v2 MDD: -0.7195172517555167


In [35]:
#Inizio calcolo paramtri base

In [36]:
periods_per_year = 2190

In [37]:
#Cagr

In [38]:
periods_per_year = 2190

def CAGR(equity_curve):
    total_return = equity_curve.iloc[-1]
    n_years = len(equity_curve) / periods_per_year
    return total_return ** (1 / n_years) - 1

cagr_bh = CAGR(df["equity_buyhold"])
cagr_strat = CAGR(df["equity_strategy_v2"])

print("CAGR Buy & Hold:", cagr_bh)
print("CAGR Strategy:", cagr_strat)

CAGR Buy & Hold: 0.38871187900254367
CAGR Strategy: 0.5276788112746382


In [39]:
#Volatilita annualizzata

In [40]:
def annual_vol(log_returns):
    return log_returns.std() * np.sqrt(periods_per_year)

vol_bh = annual_vol(df["log_return"])
vol_strat = annual_vol(df["strategy_return_v2"])

print("Vol Buy & Hold:", vol_bh)
print("Vol Strategy:", vol_strat)

Vol Buy & Hold: 0.694776901106761
Vol Strategy: 0.4826499430830003


In [41]:
#Sharpe ratio risk free = 0

In [42]:
def sharpe_ratio(log_returns):
    mean_return = log_returns.mean() * periods_per_year
    vol = annual_vol(log_returns)
    return mean_return / vol

sharpe_bh = sharpe_ratio(df["log_return"])
sharpe_strat = sharpe_ratio(df["strategy_return_v2"])

print("Sharpe Buy & Hold:", sharpe_bh)
print("Sharpe Strategy:", sharpe_strat)

Sharpe Buy & Hold: 0.47263605224741373
Sharpe Strategy: 0.8780114080528044


In [43]:
#Calmar ratio

In [44]:
mdd_bh = max_drawdown(df["equity_buyhold"])
mdd_strat = max_drawdown(df["equity_strategy_v2"])

calmar_bh = cagr_bh / abs(mdd_bh)
calmar_strat = cagr_strat / abs(mdd_strat)

print("Calmar Buy & Hold:", calmar_bh)
print("Calmar Strategy:", calmar_strat)

Calmar Buy & Hold: 0.46327074483675773
Calmar Strategy: 0.7333789564978174


In [45]:
#Featuring Engineerim=nh ETH

In [46]:
eth = pd.read_csv("../datas/eth_4h.csv")
eth["timestamp"] = pd.to_datetime(eth["timestamp"])
eth = eth.sort_values("timestamp").reset_index(drop=True)

# Log return
eth["log_return"] = np.log(eth["close"] / eth["close"].shift(1))

# ATR 14
eth["previous_close"] = eth["close"].shift(1)

eth["tr1"] = eth["high"] - eth["low"]
eth["tr2"] = abs(eth["high"] - eth["previous_close"])
eth["tr3"] = abs(eth["low"] - eth["previous_close"])

eth["true_range"] = eth[["tr1","tr2","tr3"]].max(axis=1)

eth["atr_14"] = eth["true_range"].rolling(14).mean()

# Tail target
eth["future_return"] = np.log(eth["close"].shift(-6) / eth["close"])
tail_threshold_eth = eth["future_return"].quantile(0.05)

eth["tail_event"] = (eth["future_return"] <= tail_threshold_eth).astype(int)

eth["year"] = eth["timestamp"].dt.year

eth = eth.dropna()

In [47]:
#Rolling 3Y model

In [48]:
eth["tail_score_raw"] = np.nan

years = sorted(eth["year"].unique())

for test_year in years[4:]:
    
    train_years = [test_year-3, test_year-2, test_year-1]
    
    train_data = eth[eth["year"].isin(train_years)]
    test_data = eth[eth["year"] == test_year]
    
    if len(test_data) < 200:
        continue
    
    X_train = train_data[["atr_14"]]
    y_train = train_data["tail_event"]
    
    X_test = test_data[["atr_14"]]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    
    eth.loc[eth["year"] == test_year, "tail_score_raw"] = model.predict_proba(X_test_scaled)[:,1]

In [49]:
#Smoothing + Index

In [50]:
eth["tail_score_smooth"] = eth["tail_score_raw"].ewm(span=20).mean()

eth["tail_index"] = (
    eth["tail_score_smooth"]
    .rolling(365)
    .apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1])
) * 100

In [51]:
#regime layer

In [52]:
eth["ema_200"] = eth["close"].ewm(span=200).mean()
eth["ema_200_slope"] = eth["ema_200"].diff()

def regime_state(row):
    if row["close"] > row["ema_200"] and row["ema_200_slope"] > 0:
        return "Bull"
    elif row["close"] < row["ema_200"] and row["ema_200_slope"] < 0:
        return "Bear"
    else:
        return "Neutral"

eth["regime_state"] = eth.apply(regime_state, axis=1)

eth["regime_exposure"] = 1.0
eth.loc[eth["regime_state"] == "Neutral", "regime_exposure"] = 0.6
eth.loc[eth["regime_state"] == "Bear", "regime_exposure"] = 0.3

In [53]:
#Combine tail + regime

In [54]:
eth["risk_level"] = eth["tail_index"].apply(risk_level)

eth["final_exposure"] = eth["regime_exposure"]

eth.loc[eth["risk_level"] == "Elevated", "final_exposure"] *= 0.9
eth.loc[eth["risk_level"] == "High", "final_exposure"] *= 0.7
eth.loc[eth["risk_level"] == "Extreme", "final_exposure"] *= 0.5

In [55]:
#Shift

In [56]:
eth["final_exposure_shifted"] = eth["final_exposure"].shift(1)

eth["strategy_return"] = eth["final_exposure_shifted"] * eth["log_return"]

eth["equity_strategy"] = np.exp(eth["strategy_return"].cumsum())
eth["equity_buyhold"] = np.exp(eth["log_return"].cumsum())

In [57]:
#Inizio calcolo metriche ETH

In [58]:
periods_per_year = 2190

def CAGR(equity_curve):
    total_return = equity_curve.iloc[-1]
    n_years = len(equity_curve) / periods_per_year
    return total_return ** (1 / n_years) - 1

def annual_vol(log_returns):
    return log_returns.std() * np.sqrt(periods_per_year)

def sharpe_ratio(log_returns):
    mean_return = log_returns.mean() * periods_per_year
    vol = annual_vol(log_returns)
    return mean_return / vol

def max_drawdown(series):
    cummax = series.cummax()
    drawdown = (series - cummax) / cummax
    return drawdown.min()

# === ETH METRICS ===

cagr_bh = CAGR(eth["equity_buyhold"])
cagr_strat = CAGR(eth["equity_strategy"])

vol_bh = annual_vol(eth["log_return"])
vol_strat = annual_vol(eth["strategy_return"])

sharpe_bh = sharpe_ratio(eth["log_return"])
sharpe_strat = sharpe_ratio(eth["strategy_return"])

mdd_bh = max_drawdown(eth["equity_buyhold"])
mdd_strat = max_drawdown(eth["equity_strategy"])

calmar_bh = cagr_bh / abs(mdd_bh)
calmar_strat = cagr_strat / abs(mdd_strat)

print("ETH CAGR Buy & Hold:", cagr_bh)
print("ETH CAGR Strategy:", cagr_strat)

print("ETH Vol Buy & Hold:", vol_bh)
print("ETH Vol Strategy:", vol_strat)

print("ETH Sharpe Buy & Hold:", sharpe_bh)
print("ETH Sharpe Strategy:", sharpe_strat)

print("ETH MDD Buy & Hold:", mdd_bh)
print("ETH MDD Strategy:", mdd_strat)

print("ETH Calmar Buy & Hold:", calmar_bh)
print("ETH Calmar Strategy:", calmar_strat)

ETH CAGR Buy & Hold: 0.25008578933263137
ETH CAGR Strategy: 0.5824369375134619
ETH Vol Buy & Hold: 0.87587451544105
ETH Vol Strategy: 0.5905448426850592
ETH Sharpe Buy & Hold: 0.2548449309692327
ETH Sharpe Strategy: 0.7772324239100052
ETH MDD Buy & Hold: -0.9408005648333597
ETH MDD Strategy: -0.7707396519117922
ETH Calmar Buy & Hold: 0.2658223205647502
ETH Calmar Strategy: 0.7556857053724275
